In [ ]:
import json
import pandas as pd
import joblib


In [ ]:
rf = joblib.load("fp_confidence_random_forest.pkl")
print("✅ FP confidence model loaded")


In [ ]:
with open("sample_scan_vulnerabilities.json", "r") as f:
    scan_data = json.load(f)

print(f"Total findings in scan 105: {len(scan_data)}")


In [ ]:
# Normalize vulnerability names helper
def normalize_name(name):
    return name.lower().strip()


In [ ]:
# Header / configuration related vulnerabilities
header_vulns = [
    "missing_security_header",
    "missing anti-clickjacking header",
    "content security policy (csp) header not set",
    "missing content-security-policy header",
    "strict-transport-security header not set",
    "strict-transport-security missing max-age (non-compliant with spec)",
    "missing hsts header",
    "x-content-type-options header missing",
    "missing x-content-type-options header",
    "missing x-frame-options header",
    "multiple x-frame-options header entries",
    "missing x-xss-protection header",
    "missing referrer-policy header",
    "x-aspnet-version response header",
    "server leaks information via \"x-powered-by\" http response header field(s)",
    "server leaks version information via \"server\" http response header field",
    "permissive cors policy",
    "csp: notices",
    "csp: wildcard directive",
    "csp: style-src unsafe-inline",
    "csp: script-src unsafe-inline",
    "csp: script-src unsafe-eval",
    "csp: failure to define directive with no fallback"
]

HEADER_VULNS = set(normalize_name(v) for v in header_vulns)


In [ ]:
# Injection / exploit-based vulnerabilities
injection_vulns = [
    "sql injection",
    "sql injection - mysql",
    "sql injection - mysql (time based)",
    "sql injection - sqlite (time based)",
    "sql injection - mssql (time based)",
    "cross site scripting (reflected)",
    "cross site scripting (dom based)",
    "cross site scripting (persistent)",
    "xslt injection",
    "format string error",
    "session id in url rewrite"
]

INJECTION_VULNS = set(normalize_name(v) for v in injection_vulns)


In [ ]:
print("Header vulns:", len(HEADER_VULNS))
print("Injection vulns:", len(INJECTION_VULNS))


In [ ]:
def extract_features(vuln):
    name = vuln.get("name", "").lower().strip()
    evidence = vuln.get("evidence")

    return {
        "has_evidence": 0 if evidence in [None, "", "Not Applicable"] else 1,
        "is_header_issue": int(name in HEADER_VULNS),
        "is_injection": int(name in INJECTION_VULNS)
    }


In [ ]:
scan_data

In [ ]:
results = []

for vuln in scan_data:
    features = extract_features(vuln)
    X = pd.DataFrame([features])

    fp_confidence = rf.predict_proba(X)[0][1]

    vuln_result = {
        **vuln,
        **features,
        "fp_confidence": round(fp_confidence, 3)
    }

    results.append(vuln_result)


In [ ]:
df_results = pd.DataFrame(results)
df_results.head()


In [ ]:
THRESHOLD = 0.7

df_results["fp_filtered"] = (df_results["fp_confidence"] >= THRESHOLD).astype(int)


In [ ]:
total_alerts = len(df_results)
filtered_alerts = df_results["fp_filtered"].sum()

print(f"Total alerts: {total_alerts}")
print(f"Filtered as likely FP: {filtered_alerts}")
print(f"FP Reduction: {(filtered_alerts / total_alerts) * 100:.2f}%")


In [ ]:
df_results.groupby("is_injection")["fp_filtered"].mean()


In [ ]:
pd.crosstab(df_results["severity"], df_results["fp_filtered"], normalize="index") * 100


In [ ]:
df_results.to_json(
    "sample_scan_with_fp_confidence.json",
    orient="records",
    indent=2
)

print("✅ Scan 105 results saved with FP confidence")


In [ ]:
# Load Scan 105 results WITH FP confidence
df = pd.read_json("sample_scan_with_fp_confidence.json")

print(f"Data loaded successfully. Total rows: {len(df)}")
df.head()

In [ ]:
# Load Scan 105 results WITH FP confidence
df = pd.read_json("sample_scan_with_fp_confidence.json")

print(f"Data loaded successfully. Total rows: {len(df)}")
df.head()

In [ ]:
before_count = len(df)

after_count = len(df[df["fp_filtered"] == 0])

filtered_count = len(df[df["fp_filtered"] == 1])

print(f"Before ML: {before_count} alerts")
print(f"After ML: {after_count} alerts")
print(f"Filtered: {filtered_count} alerts")
print(f"Reduction: {(filtered_count / before_count) * 100:.2f}%")


In [ ]:
# Alerts BEFORE ML (all findings)
before_df = df.copy()

# Alerts AFTER ML (kept findings)
after_df = df[df["fp_filtered"] == 0]

# Alerts FILTERED by ML (likely FP)
filtered_df = df[df["fp_filtered"] == 1]

print(f"Before ML: {len(before_df)}")
print(f"After ML: {len(after_df)}")
print(f"Filtered: {len(filtered_df)}")


In [ ]:
from IPython.display import display, HTML

def display_with_scroll(df, title, height=300):
    display(HTML(f"<h4>{title}</h4>"))
    display(HTML(
        df.to_html(index=False)
        .replace(
            "<table",
            f"<table style='display:block; max-height:{height}px; overflow-y:auto; border:1px solid #ccc;'"
        )
    ))


In [ ]:
display_with_scroll(
    before_df[[
        "name",
        "severity",
        "url",
        "has_evidence",
        "is_header_issue",
        "fp_confidence"
    ]],
    title="Before ML – All Alerts"
)


In [ ]:
display_with_scroll(
    after_df[[
        "name",
        "severity",
        "url",
        "has_evidence",
        "is_header_issue",
        "fp_confidence"
    ]],
    title="After ML – Retained Alerts"
)


In [ ]:
display_with_scroll(
    filtered_df[[
        "name",
        "severity",
        "url",
        "has_evidence",
        "is_header_issue",
        "fp_confidence"
    ]],
    title="Filtered Alerts – Likely False Positives"
)


In [ ]:
comparison_df = pd.DataFrame({
    "Before ML": before_df["name"].value_counts(),
    "After ML": after_df["name"].value_counts()
}).fillna(0).astype(int)

display_with_scroll(
    comparison_df.reset_index().rename(columns={"index": "Vulnerability Name"}),
    title="Before vs After ML – Vulnerability Counts",
    height=400
)


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import FileLink

labels = ["Before ML", "After ML"]
values = [len(before_df), len(after_df)]

plt.figure(figsize=(6, 4))
plt.bar(labels, values)
plt.ylabel("Number of Alerts")
plt.title("Alert Reduction After ML Application")
plt.tight_layout()

# Save figure
plt.savefig("alert_reduction_before_after_ml.png", dpi=300, bbox_inches="tight")
plt.show()

# Download link
FileLink("alert_reduction_before_after_ml.png")


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df["fp_confidence"], bins=20)
plt.xlabel("FP Confidence")
plt.ylabel("Count")
plt.title("FP Confidence Score Distribution")
plt.show()


In [ ]:
# Exclude 'info' severity
df_no_info = df[df["severity"].isin(["high", "medium", "low"])]

severity_before = df_no_info["severity"].value_counts()
severity_after = df_no_info[df_no_info["fp_filtered"] == 0]["severity"].value_counts()

severity_df = (
    pd.DataFrame({
        "Before ML": severity_before,
        "After ML": severity_after
    })
    .fillna(0)
)

# Ensure consistent ordering
severity_df = severity_df.reindex(["high", "medium", "low"])

severity_df.plot(kind="bar", figsize=(7,4))
plt.ylabel("Number of Alerts")
plt.title("Severity Preservation Analysis (Info Excluded)")
plt.tight_layout()

# Save for download
plt.savefig("severity_preservation_no_info.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
inj_stats = pd.crosstab(df["is_injection"], df["fp_filtered"])

inj_stats.plot(kind="bar", figsize=(6,4))
plt.xlabel("Injection Vulnerability (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.title("Injection Vulnerability Preservation")
plt.show()


In [ ]:
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
reductions = []

for t in thresholds:
    filtered = (df["fp_confidence"] >= t).sum()
    reductions.append((filtered / len(df)) * 100)

plt.figure(figsize=(6,4))
plt.plot(thresholds, reductions, marker="o")
plt.xlabel("FP Confidence Threshold")
plt.ylabel("FP Reduction (%)")
plt.title("Threshold Stability Analysis")
plt.show()
